In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

In [3]:
TRAIN_PATH = "./data/mpg_train.csv" # <- change these for new data
TEST_PATH = "./data/mpg_test.csv"
TARGET = "mpg"
ID = "id"
DROP_COLS = ["name"] # other columns that are not features (free text, too many unique values)

In [4]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train

,id,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,1,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,3,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
2,4,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
3,6,15.0,8,429.0,198.0,4341,10.0,70,usa,ford galaxie 500
4,7,14.0,8,454.0,220.0,4354,9.0,70,usa,chevrolet impala
...,...,...,...,...,...,...,...,...,...,...
313,394,27.0,4,140.0,86.0,2790,15.6,82,usa,ford mustang gl
314,395,44.0,4,97.0,52.0,2130,24.6,82,europe,vw pickup
315,396,32.0,4,135.0,84.0,2295,11.6,82,usa,dodge rampage
316,397,28.0,4,120.0,79.0,2625,18.6,82,usa,ford ranger


In [5]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 318 entries, 0 to 317
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            318 non-null    int64  
 1   mpg           318 non-null    float64
 2   cylinders     318 non-null    int64  
 3   displacement  318 non-null    float64
 4   horsepower    315 non-null    float64
 5   weight        318 non-null    int64  
 6   acceleration  318 non-null    float64
 7   model_year    318 non-null    int64  
 8   origin        318 non-null    str    
 9   name          318 non-null    str    
dtypes: float64(4), int64(4), str(2)
memory usage: 25.0 KB


In [6]:
train = train.dropna(subset=[TARGET]) # rows without a target can't be used
X = train.drop(columns=[ID, TARGET] + DROP_COLS)
y = train[TARGET]
X_sub = test.drop(columns=[ID] + DROP_COLS) # features of the data to submit

In [7]:
cat_cols = X.select_dtypes(exclude="number").columns # text columns
for c in cat_cols: # same categories for train and test, unseen -> NaN
	cats = X[c].astype("category").cat.categories
	X[c] = pd.Categorical(X[c], categories=cats)
	X_sub[c] = pd.Categorical(X_sub[c], categories=cats)

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
	X, y, test_size=0.2, random_state=42)

In [9]:
X

,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,8,307.0,130.0,3504,12.0,70,usa
1,8,318.0,150.0,3436,11.0,70,usa
2,8,304.0,150.0,3433,12.0,70,usa
3,8,429.0,198.0,4341,10.0,70,usa
4,8,454.0,220.0,4354,9.0,70,usa
...,...,...,...,...,...,...,...
313,4,140.0,86.0,2790,15.6,82,usa
314,4,97.0,52.0,2130,24.6,82,europe
315,4,135.0,84.0,2295,11.6,82,usa
316,4,120.0,79.0,2625,18.6,82,usa


In [10]:
y

0      18.0
1      18.0
2      16.0
3      15.0
4      14.0
       ... 
313    27.0
314    44.0
315    32.0
316    28.0
317    31.0
Name: mpg, Length: 318, dtype: float64

In [11]:
model = LGBMRegressor(random_state=42, verbose=-1) # no scaling, missing values OK
model.fit(X_train, y_train)

,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [12]:
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)

print(f"Train  RMSE: {root_mean_squared_error(y_train, y_pred_train):.2f}  R2: {r2_score(y_train, y_pred_train):.3f}")
print(f"Val    RMSE: {root_mean_squared_error(y_val, y_pred_val):.2f}  R2: {r2_score(y_val, y_pred_val):.3f}")

Train  RMSE: 1.75  R2: 0.952
Val    RMSE: 1.79  R2: 0.944


In [13]:
SAVE_CSV = False

submission_df = test[[ID]].copy()
submission_df[TARGET] = model.predict(X_sub)
if SAVE_CSV:
	submission_df.to_csv("submission.csv", index=False)

submission_df

,id,mpg
0,2,14.043745
1,5,17.226199
2,13,14.914066
3,16,18.387940
4,18,21.520811
...,...,...
75,379,37.586937
76,380,34.536460
77,387,23.702492
78,390,24.782541
